In [1]:
import numpy as np
import pandas as pd
import re
import warnings
data = pd.read_csv("E-commerce_cosmetic_dataset_utf8.csv")

warnings.filterwarnings('ignore')

og_df = data

def fix_dtypes(df: pd.DataFrame):
    df["noofratings"] = df["noofratings"].astype(str).apply(lambda x: re.sub(r"[^\d.-]", "", x))
    df['noofratings'] = pd.to_numeric(data['noofratings'], errors = 'coerce')

    df["size"] = df["size"].astype(str).apply(lambda x: re.sub(r"[^\d.-]", "", x))
    df['size'] = pd.to_numeric(data['size'], errors = 'coerce')

    df["rating"] = df["rating"].astype(str).apply(lambda x: re.sub(r"[^\d.-]", "", x))
    df['rating'] = pd.to_numeric(data['rating'], errors = 'coerce')

    return df
    

def drop_rows(df: pd.DataFrame):
    df = df.drop_duplicates()
    df = df[df['price'].notnull() | df['noofratings'].notnull()]
    df = df[((df['rating'] <= 5.0) & (df['rating'] >= 0.0)) | df['rating'].isnull()]
    no_size = df[(df['size'] == 0.0) & (df['form'].isin(['cream', 'gel', 'liquid']))].index
    df = df.drop(no_size)

    return df

def fill_missing_values_numerical(df: pd.DataFrame):
    df['price'] = df['price'].fillna(df['price'].median())
    df['rating'] = df['rating'].fillna(df['rating'].median())
    df['noofratings'] = df['noofratings'].fillna(df['noofratings'].median())
    df['size'] = df['size'].fillna(df['size'].median())
    return df

def remove_except_first(string, char):
    idx = string.find(char)
    if idx == -1:
        return string
    return string[:idx+1] + string[idx+1:].replace(char, '')

def fill_missing_values_categorical(df: pd.DataFrame):
    df['type'] = df['type'].fillna(df['type'].bfill())
    df['color'] = df['color'].fillna('No color')
    for sub_c in df['subcategory'].unique():
        sub_c_df = df[df['subcategory'] == sub_c]
        mode = sub_c_df['ingredients'].mode()
        cols = (df['subcategory'] == sub_c) & (df['ingredients'].isnull())
        df.loc[cols, 'ingredients'] = re.sub("[\d-]", '', str(mode))
    for col in ['product_name', 'ingredients', 'brand', 'color']:
        special_char_rows = df[(df[col].str.contains("┐¢", regex=False))]
        special_char_rows[col] = special_char_rows[col].str.replace(r'┐¢', 'è', regex=False)
        special_char_rows[col] = special_char_rows[col].str.replace(r'´|┐', '', regex=True)
        special_char_rows[col] = special_char_rows[col].apply(lambda x: remove_except_first(x, 'è'))
        special_char_rows[col] = special_char_rows[col].str.replace('\sCrè\s|Crè$|\sCCè\s', 'Crème', regex=True)
    return df

fixed_data = fix_dtypes(data)
filtered_data = drop_rows(fixed_data)
partial_filled_data = fill_missing_values_numerical(filtered_data)
cleaned_data = fill_missing_values_categorical(partial_filled_data)
cleaned_data

,product_name,website,country,category,subcategory,title-href,price,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,599.0,Carlton London,"Hydrogenated Soybean , Petrolatum, Paraffi...",aerosol,all,"Top Note: Orange Blossom, Blackberry | Heart N...",100.0,3.90,19.0
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,149.0,Charlene,"Hydrogenated Soybean , Petrolatum, Paraffi...",aerosol,all,Unit count type:,30.0,4.40,4031.0
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,298.0,Charlene,"Hydrogenated Soybean , Petrolatum, Paraffi...",aerosol,all,Unit count type:,30.0,4.40,4072.0
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,245.0,Denver,"Hydrogenated Soybean , Petrolatum, Paraffi...",aerosol,all,Long-Lasting Scent,60.0,4.20,61.0
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,422.0,Denver,"Hydrogenated Soybean , Petrolatum, Paraffi...",aerosol,all,Long-Lasting Scent,100.0,4.30,342.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12610,Bright Future - Smoothing & Brightening Concea...,Sephora,India,face,concealer,https://sephora.nnnow.com/mac-cosmetics-prep--...,1900.0,MAC Cosmetics,"Water\Aqua\Eau, Glycerin, Butylene Glycol, Toc...",cream,"Combination, Dry, Oily, Normal",Pinklite,100.0,3.15,14.0
12611,Bright Future - Smoothing & Brightening Concea...,Sephora,India,face,concealer,https://sephora.nnnow.com/mac-cosmetics-prep--...,2150.0,MAC Cosmetics,"Water\Aqua\Eau, Cyclopentasiloxane, Dimethicon...",cream,"Combination, Dry, Oily, Normal",No colour,100.0,3.13,13.0
12612,Starlaa Rosy Bronze Blush Mini,Sephora,India,face,blush,https://sephora.nnnow.com/klara-cosmetics-wome...,3040.0,Klara Cosmetics,"Talc, Ethylhexyl Palmitate, Octyldodecanol, Sy...",powder,"Combination, Oily, Normal, Dry",Selfie Queen,110.0,4.03,96.0
12613,Terra Golden Brick Red Blush Travel Size Mini,Sephora,India,face,blush,https://sephora.nnnow.com/clinique-women-clini...,2950.0,CLINIQUE,"Water\Aqua\Eau, Dimethicone, Isododecane, Buty...",liquid,"Combination, Dry, Oily, Normal",All,115.0,3.15,15.0


In [2]:
cleaned_data.to_csv("E-commerce_cosmetic_dataset_cleaned.csv")